In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)

sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
data = {
    'MonthlyCharge': [50, 80, 70, 40, 90, 60, 100, 85, 45, 75],
    'ContractLength': [12, 6, 24, 12, 3, 18, 24, 6, 12, 18],
    'Age': [30, 25, 40, 35, 28, 50, 45, 32, 29, 55],
    'Churn': [0, 1, 0, 0, 1, 0, 0, 1, 0, 0]  # 0 = No Churn, 1 = Churn
}

df = pd.DataFrame(data)
df.info()
df.describe()

In [ ]:
print("Class balance (Churn):")
print(df['Churn'].value_counts())
print()
print(df['Churn'].value_counts(normalize=True).round(2), "(proportions)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Class distribution
sns.countplot(x='Churn', data=df, ax=axes[0], palette=['#4C72B0', '#DD8452'])
axes[0].set_title('Churn Class Distribution')
axes[0].set_xticklabels(['No Churn (0)', 'Churn (1)'])

# Correlation heatmap
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()

In [ ]:
X = df[['MonthlyCharge', 'ContractLength', 'Age']]
y = df['Churn']

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print()
print("y_train:", y_train.tolist())
print("y_test :", y_test.tolist())

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled

In [ ]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("Model coefficients:", model.coef_)
print("Model intercept:", model.intercept_)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # probability of class 1 (Churn)

results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Churn Probability': y_proba.round(3)
}, index=y_test.index)

results

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2f} ({acc*100:.0f}%)")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

fig, ax = plt.subplots(figsize=(4, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0))

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")